# **Importing the needed libraries**

In [ ]:
import os, shutil, random
from google.colab import files
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D, GlobalAveragePooling2D
from tensorflow.keras.applications import ResNet50, VGG16, DenseNet121
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.models import load_model
import time
from sklearn.metrics import classification_report


# **Uploading the data using Kaggle API**

In [ ]:
!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia

In [ ]:
!unzip '*.zip' -d dataset/

# **Preparing the data**

In [ ]:
#define path
base_path = "dataset/chest_xray"
combined_path = "combined_data"
aug_path = "augmented_data"
train_path = "ready_splited/train"
test_path = "ready_splited/test"
val_path = "ready_splited/val"

In [ ]:
# Create directories
for class_name in ["NORMAL", "PNEUMONIA"]:
    os.makedirs(os.path.join(combined_path, class_name), exist_ok=True)

In [ ]:
#merging all train,test and validation into one dataset
for split in ["train", "test", "val"]: #loop over each data split

        split_path = os.path.join(base_path, split) # which path to loop through (ex"dataset/chest_xray/train", ...)

        if os.path.exists(split_path): #making sure file available

            for class_name in ["NORMAL", "PNEUMONIA"]: #loop over each class

                class_folder = os.path.join(split_path, class_name)# which path to loop through (ex"dataset/chest_xray/train/PNEUMONIA", ...)

                if os.path.exists(class_folder):#making sure file available

                    for image in os.listdir(class_folder):#loop over each image

                        src = os.path.join(class_folder, image)#current image path

                        dst = os.path.join(combined_path, class_name, image)#distination path (ex"combined_data/PNEUMONIA/image.jpeg", ...)

                        if not os.path.exists(dst): #make sure not already copied

                            shutil.copy2(src, dst) #copy image from source to destination

In [ ]:
#counting num of normal
normal_num= len(os.listdir(os.path.join(combined_path, "NORMAL")))
#counting num of pneumonia
pneumonia_num = len(os.listdir(os.path.join(combined_path, "PNEUMONIA")))

print(f"Total images: {normal_num + pneumonia_num}")
print(f"Normal: {normal_num}, Pneumonia: {pneumonia_num}")

# **Spliting the data (avoid leak when augmentation)**

In [ ]:
#creating a dictionery to define numbers of classes for each split
split_class_num= {'test': {'NORMAL': 250, 'PNEUMONIA': 250}, 'val': {'NORMAL': 100, 'PNEUMONIA': 100}}

for class_name in ['NORMAL', 'PNEUMONIA']:
    #define path of the img
    img = os.listdir(os.path.join("combined_data", class_name)) #(ex, (combined_data/NORMAL))
    # Randomly shuffle img randomly -- prevent bias when split
    random.shuffle(img)

    #number of img for test and val for the current class (normal or peneumonia)
    test = split_class_num['test'][class_name]
    val = split_class_num['val'][class_name]
    #remainning images are for training
    train = img[test + val:]

    # dictionary to map split with the correct list of images.
    split_map = {
        #begining to test
        'test': img[:test],
        #test till end of test and val
        'val': img[test:test + val],
        #remainning images are for training
        'train':img[test + val:]
    }
    # Looping through each split and its list of images.
    for split, split_imgs in split_map.items():
        #Define where images will be saved(ex, "ready_splited/test/NORMAL").
        split_path = os.path.join("ready_splited", split, class_name)
        #ensuring the folder is made if not create.
        os.makedirs(split_path, exist_ok=True)

        for img in split_imgs:
            #copy the image from original folder to new split folder
            shutil.copy2(os.path.join("combined_data", class_name, img), os.path.join(split_path, img))

**Training data**

In [ ]:
train_normal_num= len(os.listdir(os.path.join( train_path,"NORMAL")))
train_pneumonia_num= len(os.listdir(os.path.join(train_path, "PNEUMONIA")))
print(f"Total Training images: {train_normal_num + train_pneumonia_num}")
print(f"Normal: {train_normal_num}, Pneumonia: {train_pneumonia_num}")

**testing data**

In [ ]:
test_normal_num= len(os.listdir(os.path.join( test_path,"NORMAL")))
test_pneumonia_num= len(os.listdir(os.path.join(test_path, "PNEUMONIA")))
print(f"Total testing images: {test_normal_num + test_pneumonia_num}")
print(f"Normal: {test_normal_num}, Pneumonia: {test_pneumonia_num}")

**validation data**

In [ ]:
val_normal_num= len(os.listdir(os.path.join( val_path,"NORMAL")))
val_pneumonia_num= len(os.listdir(os.path.join(val_path, "PNEUMONIA")))
print(f"Total validation images: {val_normal_num + val_pneumonia_num}")
print(f"Normal: {val_normal_num}, Pneumonia: {val_pneumonia_num}")

# **Data Auggmentation only for training**

In [ ]:
# Create directories to save the augmented data in
for CN in ['NORMAL', 'PNEUMONIA']:
    os.makedirs(os.path.join(aug_path, CN), exist_ok=True)

In [ ]:
# configuration of data augmentation
data_aug_train = ImageDataGenerator(
    rotation_range=20, width_shift_range=0.2, height_shift_range=0.2,
    horizontal_flip=True, zoom_range=0.2, shear_range=0.2, fill_mode='nearest'
)

In [ ]:
img_count = 0
target_count = 3923 #same number as penumonia class to balance data

In [ ]:
# Training data with augmentation
#loop over images in normal folder only
for img_name in os.listdir(os.path.join(train_path, 'NORMAL')):

    if img_count >= target_count: #if num of images exceeds the target stop

        break

    try:

        img = Image.open(os.path.join(os.path.join(train_path, 'NORMAL'), img_name)).convert("RGB")
        img_array = np.expand_dims(np.array(img), axis=0) #expanding the image dimintation so it becomes compatble with keras

        aug_iter = data_aug_train.flow(img_array, batch_size=1)

        for i in range(min(5, target_count - img_count)): #up to five versions, stop if reached target_num

            save_path = os.path.join(os.path.join(aug_path, 'NORMAL'), f"{img_count}_{img_name}")#saving path and img file name

            Image.fromarray(next(aug_iter)[0].astype('uint8')).save(save_path)#from numpy back to

            img_count += 1

            if img_count >= target_count:

                break

    except Exception as e:   #error handling to not stop because of bad images
        print(f"Skipping {img_name}: {e}")


In [ ]:
pneumonia_src = os.path.join(train_path, 'PNEUMONIA')#source
pneumonia_dst = os.path.join(aug_path, 'PNEUMONIA')#distination

for img_name in os.listdir(os.path.join(train_path, 'PNEUMONIA')): #loop through each image from the class pneumonia in training data
    #copy image from source to didstination
    shutil.copy2(os.path.join(pneumonia_src, img_name), os.path.join(os.path.join(aug_path, 'PNEUMONIA'), img_name))

**making sure that augmentation went good**

In [ ]:
#making sure that augmentation was done
train_normal_num_aug= len(os.listdir(os.path.join(aug_path,"NORMAL")))
train_pneumonia_aug= len(os.listdir(os.path.join(aug_path, "PNEUMONIA")))
print(f"Total Training images: {train_normal_num_aug + train_pneumonia_aug}")
print(f"Normal: {train_normal_num_aug}, Pneumonia: {train_pneumonia_aug}")

In [ ]:
#downloading the data for backup
shutil.make_archive('/content/augmented_data', 'zip', '/content/augmented_data')
files.download('/content/augmented_data.zip')

# **Data generation and loading**

In [ ]:
#data loader for testing data
val_test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = val_test_datagen.flow_from_directory(
    test_path, target_size=(224, 224), color_mode='rgb',#color to match model input
    batch_size=32, class_mode='binary', shuffle=False
)

In [ ]:
#data loader for validation data
val_generator = val_test_datagen.flow_from_directory(
    val_path, target_size=(224, 224),color_mode='rgb',#color to match model input
    batch_size=32, class_mode='binary', shuffle=False
)

In [ ]:
#data loader for validation data
train_path = "augmented_data"
train_generator = val_test_datagen.flow_from_directory(
    train_path, target_size=(224, 224), color_mode='rgb', batch_size=32,class_mode='binary',shuffle=True)

# **Models initialization as methods to train on different hyperparameters easier**

**Resnet50**

In [ ]:
def create_resnet50(learning_rate, dropout_rate):
    #loading ResNet50 model that was already trained on imagenet
    #removing original classification layer, our task is differnt we dont need them, we add custom layers then.
    #input shape is "rgb" because it was trained on 3 channel so only accepts it
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

    #freezes so weights dont change during training -transfer learning-
    base_model.trainable = False

    model = Sequential([
        base_model,
        GlobalAveragePooling2D(), #convert FM to 1d vec
        Dropout(dropout_rate),
        Dense(128, activation='relu'), #fully connected layer
        Dropout(dropout_rate),
        Dense(1, activation='sigmoid') #binary classification using sigmoid as activation
    ])

    model.compile(optimizer=Adam(learning_rate=learning_rate), #adam optimizer
                  loss='binary_crossentropy', metrics=['accuracy']) #loss binary_crossentropy due to binary classification
    return model

**VGG16**

In [ ]:
def create_vgg16(learning_rate, dropout_rate):
    #loading VGG16 model that was already trained on imagenet
    #removing original classification layer, our task is differnt we dont need them, we add custom layers then.
    #input shape is "rgb" because it was trained on 3 channel so only accepts it
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

    #freezes so weights dont change during training -transfer learning-
    base_model.trainable = False

    model = Sequential([
        base_model,
        Flatten(), #convert FM to 1d vec
        Dropout(dropout_rate),
        Dense(256, activation='relu'), #fully connected layer
        Dropout(dropout_rate),
        Dense(1, activation='sigmoid') #binary classification using sigmoid as activation
    ])

    model.compile(optimizer=Adam(learning_rate=learning_rate),#adam optimizer
                  loss='binary_crossentropy', metrics=['accuracy'])#loss binary_crossentropy due to binary classification
    return model

**Densnet121**

In [ ]:
def create_densenet121(learning_rate, dropout_rate):
    #loading DenseNet121 model that was already trained on imagenet
    #removing original classification layer, our task is differnt we dont need them, we add custom layers then.
    #input shape is "rgb" because it was trained on 3 channel so only accepts it
    base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

    #freezes so weights dont change during training -transfer learning-
    base_model.trainable = False

    model = Sequential([
        base_model,
        GlobalAveragePooling2D(), #convert FM to 1d vec
        Dropout(dropout_rate),
        Dense(128, activation='relu'),  #fully connected layer
        Dropout(dropout_rate),
        Dense(1, activation='sigmoid') #binary classification using sigmoid as activation
    ])

    model.compile(optimizer=Adam(learning_rate=learning_rate), #adam optimize
                  loss='binary_crossentropy', metrics=['accuracy']) #loss binary_crossentropy due to binary classification
    return model

**Alexnet**

In [ ]:
def create_alexnet(learning_rate, dropout_rate):
    model = Sequential([
        #conv layer 1: k =96, kernel =11x11, s= 4 (downsampling).
        Conv2D(96, (11, 11), strides=4, activation='relu', input_shape=(224, 224, 3)),
        #pooling filter (3x3), s = 2
        MaxPooling2D((3, 3), strides=2),
        #conv layer 2: k =256, kernel =5x5, padding = input size= output size.
        Conv2D(256, (5, 5), padding='same', activation='relu'),
        #pooling filter (3x3), s = 2
        MaxPooling2D((3, 3), strides=2),
        #conv layer 3: k =384, kernel =3x3, padding = input size= output size.
        Conv2D(384, (3, 3), padding='same', activation='relu'),
        #conv layer 4: k =384, kernel =3x3, padding = input size= output size.
        Conv2D(384, (3, 3), padding='same', activation='relu'),
        #conv layer 5: k =384, kernel =3x3, padding = input size= output size.
        Conv2D(256, (3, 3), padding='same', activation='relu'),
        #pooling filter (3x3), s = 2
        MaxPooling2D((3, 3), strides=2),
        #convert to 1d vec to pass to fully connected layers
        Flatten(),

        Dropout(dropout_rate),
        #huge fully connected layers n = 4096
        Dense(4096, activation='relu'), Dropout(dropout_rate),
        Dense(4096, activation='relu'), Dropout(dropout_rate),

        #binary classification using sigmoid as activation
        Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer=Adam(learning_rate=learning_rate), #adam optimize
                  loss='binary_crossentropy', metrics=['accuracy']) #loss binary_crossentropy due to binary classification
    return model

# **Preparing the parameters and functions to train the models**

**preparing the data for StratifiedKFold**

In [ ]:
X_images = [] #store image data
y_label = [] #store label (0-NORMAL, 1-PNEUMONIA)

In [ ]:
#loop through both classes
#enumerate to give normal 0 and pneumonia 1
for L, CN in enumerate(['NORMAL', 'PNEUMONIA']):
    #loop through each image in class folder
    for img_name in os.listdir(os.path.join(aug_path, CN)): #augmented_data/Normal
        #define each image path
        img_path = os.path.join(os.path.join(aug_path, CN), img_name) #augmented_data/Normal/img

        try:
            #load img and resize
            img = load_img(img_path, target_size=(224, 224))

            # convert to NumPy, normalizes pixel (0, 1)
            img_array = img_to_array(img) / 255.0

            #add image to the list
            X_images.append(img_array)
            #add label to the list
            y_label.append(L)

        except Exception as e: #error handling to not stop because of bad images

            print(f"Skipping {img_name}: {e}")


In [ ]:
#convert to numpy
X_images = np.array(X_images)
y_label = np.array(y_label)

In [ ]:
def all_folds_learning_curves(fold_histories, title="Learning Curves Across All Folds"):
    plt.figure(figsize=(14, 10))

    plt.subplot(2, 2, 1)
    for i, history in enumerate(fold_histories):
        plt.plot(history["accuracy"], label=f"Fold {i+1}")
    plt.title(f"{title} - Training Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()


    plt.subplot(2, 2, 2)
    for i, history in enumerate(fold_histories):
        plt.plot(history["val_accuracy"], label=f"Fold {i+1}")
    plt.title(f"{title} - Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()


    plt.subplot(2, 2, 3)
    for i, history in enumerate(fold_histories):
        plt.plot(history["loss"], label=f"Fold {i+1}")
    plt.title(f"{title} - Training Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()


    plt.subplot(2, 2, 4)
    for i, history in enumerate(fold_histories):
        plt.plot(history["val_loss"], label=f"Fold {i+1}")
    plt.title(f"{title} - Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
def Best_learning_curve(history):
    plt.figure(figsize=(12, 5))


    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Val Accuracy')
    plt.title('Accuracy over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()


    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title('Loss over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
#initialization of cross validation using StratifiedKFold
cross_validation = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [ ]:
#define hyperparameter combinations
hyperparameters = [
    {'learning_rate': 0.01, 'dropout_rate': 0.4},
    {'learning_rate': 0.001, 'dropout_rate': 0.2},
    {'learning_rate': 0.005, 'dropout_rate': 0.3}
]

# **Models training and best models validation**

# **ResNet50**

In [ ]:
#creating a list to save all models result in
Resnet50_model_results = []

In [ ]:
#loop over each combinations
for parameters in hyperparameters:
    #logging start time to calculate time complexity
    ST = time.time()

    print(f"\n Hyperparameters: {parameters}")

    f = 1#track fold num
    training_trackers = [] #to save train accuracy and calculate mean of the folds
    validation_trackers = [] #to save validation accuracy and calculate mean of the folds
    fold_trackers = [] #to save each fold performance to plot


    #loop overk-fold splits
    for train_idx, val_idx in cross_validation.split(X_images, y_label):

        print(f"\n Fold: {f}")

        #split the data for the current fold
        X_train, X_val = X_images[train_idx], X_images[val_idx]
        y_train, y_val = y_label[train_idx], y_label[val_idx]

        #combiling the model using the method created and passing the current hyperparameter combo
        resnet = create_resnet50(learning_rate=parameters["learning_rate"],dropout_rate=parameters["dropout_rate"])

        #save best model to load it in testing
        model_name = f"Resnet50_fold{f}_lr-{parameters['learning_rate']}_dr-{parameters['dropout_rate']}.h5"
        checkpoint = ModelCheckpoint(model_name, monitor='val_accuracy', save_best_only=True, verbose=0)

        #patience=3 , meaning if accuracy didn't improve for 3 epochs stop
        #restore_best_weights=True, meaning when it stops it would take the best epochs weights not the last ones.
        callback = [EarlyStopping(patience=3, restore_best_weights=True), checkpoint]

        #fit the model
        h = resnet.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=7,
            batch_size=32,
            callbacks=callback,
            verbose=0
        )

        #evaluate
        scores = resnet.evaluate(X_val, y_val, verbose=0)
        #add the scores to the trackers to calculate mean
        training_trackers.append(h.history["accuracy"][-1])
        validation_trackers.append(scores[1])
        #to plot
        fold_trackers.append(h.history)

        f += 1
    #calculating the time taken to train this model
    ET = time.time()
    Overall = (ET - ST) / 60  # to minutes
    print(f" Time taken to train this model: {Overall:.2f} minutes")

    #plot learning curve for all folds
    all_folds_learning_curves(fold_trackers, title=f"lr={parameters['learning_rate']}  dr={parameters['dropout_rate']}")

    #adding scores to the list to better compare
    Resnet50_model_results.append({
        "hyperparameters": parameters,
        "training_accuracy": np.mean(training_trackers),
        "validation_accuracy": np.mean(validation_trackers),
        "Training time": Overall,
    })

In [ ]:
Resnet50_model_results= pd.DataFrame(Resnet50_model_results)

In [ ]:
Resnet50_model_results

In [ ]:
#find best model
best_model_Resnet50 = Resnet50_model_results.loc[Resnet50_model_results["validation_accuracy"].idxmax()]
print("\n Best ResNet50 Model:")
print("Hyperparameters:", best_model_Resnet50["hyperparameters"])
print(f"Training Accuracy: {best_model_Resnet50['training_accuracy']:.4f}")
print(f"Validation Accuracy: {best_model_Resnet50['validation_accuracy']:.4f}")

**Training and Saving the best performing model using the validation data**

In [ ]:
best_model_res = create_resnet50(learning_rate=0.001, dropout_rate=0.2)
best_model_res.summary()
ST = time.time()
history = best_model_res.fit(
    train_generator,
    validation_data=val_generator,
    epochs=12,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=1
)
Best_learning_curve(history)

ET = time.time()
Overall = (ET - ST) / 60  # to minutes
print(f" Time taken to train this model: {Overall:.2f} minutes")

train_loss, train_accuracy = best_model_res.evaluate(train_generator, verbose=1)
print(f"Training Loss: {train_loss:.4f}")
print(f"Training Accuracy: {train_accuracy:.4f}")

val_loss, val_accuracy = best_model_res.evaluate(val_generator, verbose=1)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

best_model_res.save("ResNet50_best.keras")

# **VGG16**

In [ ]:
#creating a list to save all models result in
vgg16_model_results = []

In [ ]:
#loop over each combinations
for parameters in hyperparameters:

    #logging start time to calculate time complexity
    ST = time.time()

    print(f"\n Hyperparameters: {parameters}")

    f = 1#track fold num
    training_trackers = [] #to save train accuracy and calculate mean of the folds
    validation_trackers = [] #to save validation accuracy and calculate mean of the folds
    fold_trackers = [] #to save each fold performance to plot


    #loop overk-fold splits
    for train_idx, val_idx in cross_validation.split(X_images, y_label):

        print(f"\n Fold: {f}")

        #split the data for the current fold
        X_train, X_val = X_images[train_idx], X_images[val_idx]
        y_train, y_val = y_label[train_idx], y_label[val_idx]

        #combiling the model using the method created and passing the current hyperparameter combo
        vgg16 = create_vgg16(learning_rate=parameters["learning_rate"],dropout_rate=parameters["dropout_rate"])


        #save best model to load it in testing
        model_name = f"VGG16_fold{f}_lr-{parameters['learning_rate']}_dr-{parameters['dropout_rate']}.h5"
        checkpoint = ModelCheckpoint(model_name, monitor='val_accuracy', save_best_only=True, verbose=0)

        #patience=3 , meaning if accuracy didn't improve for 3 epochs stop
        #restore_best_weights=True, meaning when it stops it would take the best epochs weights not the last ones.
        callback = [EarlyStopping(patience=3, restore_best_weights=True), checkpoint]

        #fit the model
        h = vgg16.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=7,
            batch_size=32,
            callbacks=callback,
            verbose=0
        )

        #evaluate
        scores = vgg16.evaluate(X_val, y_val, verbose=0)
        #add the scores to the trackers to calculate mean
        training_trackers.append(h.history["accuracy"][-1])
        validation_trackers.append(scores[1])
        #to plot
        fold_trackers.append(h.history)

        f += 1

    #calculating the time taken to train this model
    ET = time.time()
    Overall = (ET - ST) / 60  # to minutes
    print(f" Time taken to train this model: {Overall:.2f} minutes")

    #plot learning curve for all folds
    all_folds_learning_curves(fold_trackers, title=f"lr={parameters['learning_rate']}  dr={parameters['dropout_rate']}")

    #adding scores to the list to better compare
    vgg16_model_results.append({
        "hyperparameters": parameters,
        "training_accuracy": np.mean(training_trackers),
        "validation_accuracy": np.mean(validation_trackers),
        "Training time": Overall,
        "model_name": model_name
    })

In [ ]:
vgg16_model_results= pd.DataFrame(vgg16_model_results)

In [ ]:
vgg16_model_results

In [ ]:
#find best model
best_model_Vgg16 = vgg16_model_results.loc[vgg16_model_results["validation_accuracy"].idxmax()]
print("\n Best VGG16 Model:")
print("Hyperparameters:", best_model_Vgg16["hyperparameters"])
print(f"Training Accuracy: {best_model_Vgg16['training_accuracy']:.4f}")
print(f"Validation Accuracy: {best_model_Vgg16['validation_accuracy']:.4f}")
print(f"Model File to save: {best_model_Vgg16['model_name']}")

**Training and Saving the best performing model using the validation data**

In [ ]:
best_model_vgg = create_vgg16(learning_rate=0.005, dropout_rate=0.3)
best_model_vgg.summary()
ST = time.time()
history = best_model_vgg.fit(
    train_generator,
    validation_data=val_generator,
    epochs=12,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=1
)
Best_learning_curve(history)

ET = time.time()
Overall = (ET - ST) / 60  # to minutes
print(f" Time taken to train this model: {Overall:.2f} minutes")

train_loss, train_accuracy = best_model_vgg.evaluate(train_generator, verbose=1)
print(f"Training Loss: {train_loss:.4f}")
print(f"Training Accuracy: {train_accuracy:.4f}")

val_loss, val_accuracy = best_model_vgg.evaluate(val_generator, verbose=1)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

best_model_vgg.save("VGG16_best.keras")

# **AlexNet**

In [ ]:
alexnet_model_results = []

In [ ]:
#loop over each combinations
for parameters in hyperparameters:
    #logging start time to calculate time complexity
    ST = time.time()

    print(f"\n Hyperparameters: {parameters}")

    f = 1#track fold num
    training_trackers = [] #to save train accuracy and calculate mean of the folds
    validation_trackers = [] #to save validation accuracy and calculate mean of the folds
    fold_trackers = [] #to save each fold performance to plot


    #loop overk-fold splits
    for train_idx, val_idx in cross_validation.split(X_images, y_label):

        print(f"\n Fold: {f}")

        #split the data for the current fold
        X_train, X_val = X_images[train_idx], X_images[val_idx]
        y_train, y_val = y_label[train_idx], y_label[val_idx]

        #combiling the model using the method created and passing the current hyperparameter combo
        Alexnet = create_alexnet(learning_rate=parameters["learning_rate"],dropout_rate=parameters["dropout_rate"])

        #save best model to load it in testing
        model_name = f"Alexnet_fold{f}_lr-{parameters['learning_rate']}_dr-{parameters['dropout_rate']}.h5"
        checkpoint = ModelCheckpoint(model_name, monitor='val_accuracy', save_best_only=True, verbose=0)

        #patience=3 , meaning if accuracy didn't improve for 3 epochs stop
        #restore_best_weights=True, meaning when it stops it would take the best epochs weights not the last ones.
        callback = [EarlyStopping(patience=3, restore_best_weights=True), checkpoint]

        #fit the model
        h = Alexnet.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=7,
            batch_size=32,
            callbacks=callback,
            verbose=0
        )

        #evaluate
        scores = Alexnet.evaluate(X_val, y_val, verbose=0)
        #add the scores to the trackers to calculate mean
        training_trackers.append(h.history["accuracy"][-1])
        validation_trackers.append(scores[1])
        #to plot
        fold_trackers.append(h.history)

        f += 1

    #calculating the time taken to train this model
    ET = time.time()
    Overall = (ET - ST) / 60  # to minutes
    print(f" Time taken to train this model: {Overall:.2f} minutes")

    #plot learning curve for all folds
    all_folds_learning_curves(fold_trackers, title=f"lr={parameters['learning_rate']}  dr={parameters['dropout_rate']}")

    #adding scores to the list to better compare
    alexnet_model_results.append({
        "hyperparameters": parameters,
        "training_accuracy": np.mean(training_trackers),
        "validation_accuracy": np.mean(validation_trackers),
        "Training time": Overall,
        "model_name": model_name
    })

In [ ]:
alexnet_model_results= pd.DataFrame(alexnet_model_results)

In [ ]:
alexnet_model_results

In [ ]:
#find best model
best_model_alexnet = alexnet_model_results.loc[alexnet_model_results["validation_accuracy"].idxmax()]
print("\n Best Alexnet Model:")
print("Hyperparameters:", best_model_alexnet["hyperparameters"])
print(f"Training Accuracy: {best_model_alexnet['training_accuracy']:.4f}")
print(f"Validation Accuracy: {best_model_alexnet['validation_accuracy']:.4f}")
print(f"Model File to save: {best_model_alexnet['model_name']}")

**Training and Saving the best performing model using the validation data**

In [ ]:
best_model_alex = create_alexnet(learning_rate=0.001, dropout_rate=0.2)
best_model_alex.summary()
ST = time.time()
history = best_model_alex.fit(
    train_generator,
    validation_data=val_generator,
    epochs=12,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=1
)
Best_learning_curve(history)

ET = time.time()
Overall = (ET - ST) / 60  # to minutes
print(f" Time taken to train this model: {Overall:.2f} minutes")

train_loss, train_accuracy = best_model_alex.evaluate(train_generator, verbose=1)
print(f"Training Loss: {train_loss:.4f}")
print(f"Training Accuracy: {train_accuracy:.4f}")

val_loss, val_accuracy = best_model_alex.evaluate(val_generator, verbose=1)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

best_model_alex.save("AlexNet_best.keras")

# **DenseNet121**

In [ ]:
#creating a list to save all models result in
densenet_model_results = []

In [ ]:
#loop over each combinations
for parameters in hyperparameters:
    #logging start time to calculate time complexity
    ST = time.time()

    print(f"\n Hyperparameters: {parameters}")

    f = 1#track fold num
    training_trackers = [] #to save train accuracy and calculate mean of the folds
    validation_trackers = [] #to save validation accuracy and calculate mean of the folds
    fold_trackers = [] #to save each fold performance to plot


    #loop overk-fold splits
    for train_idx, val_idx in cross_validation.split(X_images, y_label):

        print(f"\n Fold: {f}")

        #split the data for the current fold
        X_train, X_val = X_images[train_idx], X_images[val_idx]
        y_train, y_val = y_label[train_idx], y_label[val_idx]

        #combiling the model using the method created and passing the current hyperparameter combo
        Densenet = create_densenet121(learning_rate=parameters["learning_rate"],dropout_rate=parameters["dropout_rate"])

        #save best model to load it in testing
        model_name = f"DenseNet121_fold{f}_lr-{parameters['learning_rate']}_dr-{parameters['dropout_rate']}.h5"
        checkpoint = ModelCheckpoint(model_name, monitor='val_accuracy', save_best_only=True, verbose=0)

        #patience=3 , meaning if accuracy didn't improve for 3 epochs stop
        #restore_best_weights=True, meaning when it stops it would take the best epochs weights not the last ones.
        callback = [EarlyStopping(patience=3, restore_best_weights=True), checkpoint]

        #fit the model
        h = Densenet.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=7,
            batch_size=32,
            callbacks=callback,
            verbose=0
        )

        #evaluate
        scores = Densenet.evaluate(X_val, y_val, verbose=0)
        #add the scores to the trackers to calculate mean
        training_trackers.append(h.history["accuracy"][-1])
        validation_trackers.append(scores[1])
        #to plot
        fold_trackers.append(h.history)

        f += 1

    #calculating the time taken to train this model
    ET = time.time()
    Overall = (ET - ST) / 60  # to minutes
    print(f" Time taken to train this model: {Overall:.2f} minutes")

    #plot learning curve for all folds
    all_folds_learning_curves(fold_trackers, title=f"lr={parameters['learning_rate']}  dr={parameters['dropout_rate']}")

    #adding scores to the list to better compare
    densenet_model_results.append({
        "hyperparameters": parameters,
        "training_accuracy": np.mean(training_trackers),
        "validation_accuracy": np.mean(validation_trackers),
        "Training time": Overall,
        "model_name": model_name
    })

In [ ]:
densenet_model_results= pd.DataFrame(densenet_model_results)

In [ ]:
densenet_model_results

In [ ]:
#find best model
best_model_densenet = densenet_model_results.loc[densenet_model_results["validation_accuracy"].idxmax()]
print("\n Best densenet Model:")
print("Hyperparameters:", best_model_densenet["hyperparameters"])
print(f"Training Accuracy: {best_model_densenet['training_accuracy']:.4f}")
print(f"Validation Accuracy: {best_model_densenet['validation_accuracy']:.4f}")
print(f"Model File to save: {best_model_densenet['model_name']}")

**Training and Saving the best performing model using the validation data**

In [ ]:
best_model_dense = create_densenet121(learning_rate=0.001, dropout_rate=0.2)
best_model_dense.summary()
ST = time.time()
history = best_model_dense.fit(
    train_generator,
    validation_data=val_generator,
    epochs=12,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=1
)
Best_learning_curve(history)

ET = time.time()
Overall = (ET - ST) / 60  # to minutes
print(f" Time taken to train this model: {Overall:.2f} minutes")

train_loss, train_accuracy = best_model_dense.evaluate(train_generator, verbose=1)
print(f"Training Loss: {train_loss:.4f}")
print(f"Training Accuracy: {train_accuracy:.4f}")

val_loss, val_accuracy = best_model_dense.evaluate(val_generator, verbose=1)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

best_model_dense.save("DenseNet121_best.keras")

# **Evaluation of best models of each architecture using test data**

In [ ]:
#creating method to avoid duplicating the code
def evaluate_on_test(model, model_name):
    #start from the beginning of the data
    test_generator.reset()
    #predict we use test_generator to load images batch-by-batch and applies the initialized preprocessing
    predictions = model.predict(test_generator)
    #threshold where if predection> 0.5 will return 1 (Pneumonia) not return 0 (Normal).
    y_pred = (predictions > 0.5).astype(int).flatten() #threshold where if predection> 0.5 will return 1 (Pneumonia) not return 0 (Normal).
    #actual label
    print(predictions[:10])
    y_true = test_generator.classes
    #calculate the metrices and return them
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1_score': f1_score(y_true, y_pred),
        'confusion_matrix': confusion_matrix(y_true, y_pred)
    }


In [ ]:
#loading the best models of each Architicture
best_models = {
    "ResNet50": load_model("ResNet50_best.keras"),
    "VGG16": load_model("VGG16_best.keras"),
    "AlexNet": load_model("AlexNet_best.keras"),
    "DenseNet": load_model("DenseNet121_best.keras")
}


In [ ]:
#evaluating the model and adding metrices to a list
Testing = {}
for model_name, model in best_models.items():
    print(f"\nEvaluating {model_name}...")
    results = evaluate_on_test(model, model_name)
    Testing[model_name] = results

In [ ]:
# make results into dataframe
results_df = pd.DataFrame([
    {
        'Model': model_name,
        'Accuracy': f"{results['accuracy']:.4f}",
        'Precision': f"{results['precision']:.4f}",
        'Recall': f"{results['recall']:.4f}",
        'F1-Score': f"{results['f1_score']:.4f}"
    }
    for model_name, results in Testing.items()
])

In [ ]:
results_df

In [ ]:
#confusion matrix
for model_name, results in Testing.items():
    cm = results['confusion_matrix']
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['NORMAL', 'PNEUMONIA'],
                yticklabels=['NORMAL', 'PNEUMONIA'])
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.show()

In [ ]:
#downloading the best models files to use them in the interface
files.download('AlexNet_best.keras')
files.download('DenseNet121_best.keras')
files.download('ResNet50_best.keras')
files.download('VGG16_best.keras')